# KolektorSDD on GPU: deep PatchCore (anomalib) and YOLO segmentation (Google Colab)

Local CPU results (`ml/reports/surface_defects.json`): the oriented crack filter reaches ROC-AUC 0.89; the handcrafted-feature patch bank fails (0.54) because HOG cannot separate cracks from surface grain.
This notebook runs the two methods that need a GPU and pretrained networks, using the **same split** (every 3rd product folder = test) so the numbers are comparable.

1. Runtime → Change runtime type → **T4 GPU**
2. Upload `KolektorSDD.zip` (from data.vicos.si, CC BY-NC-SA 4.0)
3. Run all; copy the metrics into `ml/reports/surface_defects_gpu.json`

In [ ]:
!pip -q install anomalib ultralytics opencv-python-headless scikit-learn

In [ ]:
from google.colab import files
files.upload()  # KolektorSDD.zip
!unzip -q -o KolektorSDD.zip -d ksdd

In [ ]:
import cv2, glob, os, shutil, numpy as np
folders = sorted(glob.glob('ksdd/kos*'))
test_folders = set(folders[::3])
items = []
for f in folders:
    for img in sorted(glob.glob(f + '/Part*.jpg')):
        mask = cv2.imread(img.replace('.jpg', '_label.bmp'), 0)
        items.append((img, mask, f in test_folders, bool((mask > 0).any())))
print(len(items), sum(t for *_, t, _ in [(0, 0, i[2], i[3]) for i in items]), 'test images')

## 1. PatchCore with pretrained WideResNet-50 features (anomalib)

In [ ]:
# Folder layout expected by anomalib: train/good, test/good, test/defect, ground_truth/defect
root = 'ksdd_anomalib'
for d in ['train/good', 'test/good', 'test/defect', 'ground_truth/defect']:
    os.makedirs(f'{root}/{d}', exist_ok=True)
for img, mask, is_test, bad in items:
    name = img.replace('/', '_')
    if not is_test and not bad:
        shutil.copy(img, f'{root}/train/good/{name}')
    elif is_test:
        shutil.copy(img, f'{root}/test/{"defect" if bad else "good"}/{name}')
        if bad:
            cv2.imwrite(f'{root}/ground_truth/defect/{name.replace(".jpg", "_mask.png")}', (mask > 0).astype('uint8') * 255)

In [ ]:
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine
dm = Folder(name='ksdd', root=root, normal_dir='train/good', abnormal_dir='test/defect', normal_test_dir='test/good', mask_dir='ground_truth/defect')
model = Patchcore(backbone='wide_resnet50_2', coreset_sampling_ratio=0.1)
engine = Engine()
engine.fit(model=model, datamodule=dm)
engine.test(model=model, datamodule=dm)  # reports image AUROC and pixel AUROC

## 2. YOLO segmentation (supervised: uses the defect masks)

In [ ]:
yroot = 'ksdd_yolo'
for split in ['train', 'val']:
    os.makedirs(f'{yroot}/images/{split}', exist_ok=True); os.makedirs(f'{yroot}/labels/{split}', exist_ok=True)
for img, mask, is_test, bad in items:
    split = 'val' if is_test else 'train'
    name = img.replace('/', '_').replace('.jpg', '')
    shutil.copy(img, f'{yroot}/images/{split}/{name}.jpg')
    h, w = mask.shape
    lines = []
    contours, _ = cv2.findContours((mask > 0).astype('uint8'), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if len(c) >= 3:
            pts = ' '.join(f'{x / w:.5f} {y / h:.5f}' for x, y in c[:, 0, :])
            lines.append(f'0 {pts}')
    open(f'{yroot}/labels/{split}/{name}.txt', 'w').write('
'.join(lines))
open('ksdd.yaml', 'w').write(f'path: {os.path.abspath(yroot)}
train: images/train
val: images/val
names:
  0: crack
')

In [ ]:
from ultralytics import YOLO
yolo = YOLO('yolo11s-seg.pt')
yolo.train(data='ksdd.yaml', imgsz=640, epochs=80, batch=16, patience=20)
metrics = yolo.val()
print(metrics.seg.map50, metrics.box.map50)

In [ ]:
# Export for CPU / edge inference in the RootCause vision module
yolo.export(format='onnx', imgsz=640)